In [3]:
import requests
import json
# Initialize Qdrant client

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, Range
from qdrant_client.http import models
import pandas as pd
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall 


In [26]:
import requests
import json

class OLLAMA:
    def __init__(self, model_name, api_endpoint='http://localhost:11434/api/generate', **kwargs):
        self.model_name = model_name
        self.api_endpoint = api_endpoint
        self.session = requests.Session()
        self.kwargs = {"temperature": 0.7, "n": 1, **kwargs}
        self.run_config = None  # Placeholder for run configuration details
        print(f"Initialized OLLAMA with model_name: {model_name}, api_endpoint: {api_endpoint}, kwargs: {self.kwargs}")

    def predict(self, question, **kwargs):
        complete_response = ""
        payload = {'model': self.model_name, 'prompt': question, **self.kwargs, **kwargs}
        with self.session.post(self.api_endpoint, json=payload, stream=True) as r:
            if r.status_code == 200:
                for line in r.iter_lines():
                    if line:
                        decoded_line = line.decode('utf-8')
                        json_response = json.loads(decoded_line)
                        complete_response += json_response.get("response", "")
                        if json_response.get("done", False):
                            break
            else:
                print(f"Error: Received status code {r.status_code}")
        return complete_response.strip()

    def set_run_config(self, run_config):
        self.run_config = run_config  # You can implement configuration adjustments based on run_config if needed

# Example usage
ollama_model = OLLAMA(model_name="mistral")
#response = ollama_model.predict("Sample question?")
#print("Response:", response)


Initialized OLLAMA with model_name: mistral, api_endpoint: http://localhost:11434/api/generate, kwargs: {'temperature': 0.7, 'n': 1}


In [22]:
client = QdrantClient(host='localhost', port=6333)

# Function to fetch document content using metadata filtering
def fetch_and_format_documents(client, collection_name):
    # Fetch documents with a dummy vector
    query_vector = np.random.rand(4096)  # Ensure dimensionality matches your Qdrant configuration
    hits = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=1  # Adjust based on the needs
    )
    # Extract and format document contents
    documents = []
    for hit in hits:
        # Assuming 'payload' and 'text' hold the document content; adjust as per your actual data structure
        document_text = hit.payload['text']
        documents.append(document_text.replace('\n', ' ').strip())
    
    return " ".join(documents)  # Join all documents into a single string if there are multiple

# Example usage
documents = fetch_and_format_documents(client, 'oratio2')
documents

'CIRCULAIRE AUX INT ERMEDIAIRES  AGR EES N°87-37 D U 24 SEPTE MBR E 1987   ∗  OBJET : Comptes spéciaux en  devises et en dina rs  conv ertibles.   L\'article  25 nouveau du décret  n°77 -608 du 27  juillet  1977 fixant  les modalités  d\'application du code  des changes di spense  de l\'obligation de cess ion l es  devises provenant des  revenus ou produits des avoirs à  l\'étranger  et des  avoi rs en devises à l\'étranger  déclarés  à la Banque Centrale  de Tunisie confor mém ent aux  articles 16 et 18 du code des changes et à l\'article 16 de  la loi n°86 -83 du 1er septembre 1986 port ant loi de  finances recti ficative pour l\'année 1986.  Ces devises peuvent être logées dans des co mptes  spéciaux  en  dev ises  ou  en  dinars  conver tibles  et  peuvent être librement utilisées  en Tunisie  et à  l\'étranger.   La présente Circulaire précise les catégories de  bénéficiaires  de ces comptes et les modalités de  déclaration à la Banque C entrale de Tunisie des avoi rs  à l\'étrang

In [8]:
from ragas import evaluate  # Update this import based on your actual library

from datasets import Dataset

# Example questions and their expected ground truth answers
questions = [
    "Quelle est la date de la circulaire mentionnée?",
    "Quelles sont les nouvelles directives pour la mise en œuvre?",
]
# Expected ground truth answers
ground_truths = [
    "La circulaire est datée du 24 septembre 1987.",
    "Les nouvelles directives stipulent que toutes les unités doivent suivre les procédures mises à jour."
]

responses = []
for question in questions:
    prompt = f"Context: {documents}\nQuestion: {question}\nAnswer:"
    response = ollama_model.predict(prompt)
    responses.append(response)


In [12]:
responses

['The text provided does not specify a date for the circular mentioned. It would be necessary to check the document itself or any additional information that might provide this detail.',
 'Les nouvelles directives pour la mise en œuvre sont les suivantes :\n\n1) Des sommes provenant de la clôture d\' un compte étranger en devises ou en dinars convertis peuvent être créditées sans autorisation préalable. Il s\'agit des montants provenant du titulaire du compte ou d\'un autre compte spécial en devises ou en dinars convertibles, ainsi que des intérêts produits par ces sommes déposées dans le compte, calculés dans les conditions fixées pour les comptes étrangers en dinars convertibles.\n\n2) Toute autre inscription au crédit du compte est soumise à l\'autorisation de la Banque Centrale de Tunisie.\n\n3) Les comptes spéciaux en "dinars convertibles" peuvent être débités sans autorisation préalable :\n   - Pour tout règlementement en Tunisie,\n   - En vue de l\'achat de toutes devises étrang

In [10]:
data = {
    "question": questions,
    "contexts":  [[documents] for _ in questions],  # Use the same flattened context for each question
    "answer": responses,
    "ground_truth": ground_truths
}

from datasets import Dataset, Features, Value, Sequence

features = Features({
    'question': Value('string'),
    'contexts': Sequence(Value('string')),  # Ensuring contexts is a sequence of strings
    'answer': Value('string'),
    'ground_truth': Sequence(Value('string'))
})

dataset = Dataset.from_dict(data)
dataset

Dataset({
    features: ['question', 'contexts', 'answer', 'ground_truth'],
    num_rows: 2
})

In [11]:
from langchain_core.prompts import PromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context","question"]
  )

In [17]:
from langchain_community.llms.ollama import Ollama

# Connect to Ollama using the container name
ollama_client = Ollama(model="mistral")


In [38]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="Mistral-7B-Instruct-v0.3",
    temperature=0,
    max_retries=2,
    # other params...
)

In [40]:
from langchain_experimental.llms.ollama_functions import OllamaFunctions

llm = OllamaFunctions(model="mistral")

c:\Users\Tifa\Desktop\rag\.venv\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `OllamaFunctions` was deprecated in LangChain 0.0.64 and will be removed in 0.4.0. An updated version of the class exists in the langchain-ollama package and should be used instead. To use it run `pip install -U langchain-ollama` and import as `from langchain_ollama import ChatOllama`.
  warn_deprecated(


In [42]:
evaluation_results = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])


AttributeError: 'OLLAMA' object has no attribute 'set_run_config'

In [31]:
import requests
import json

class OLLAMA:
    def __init__(self, model_name, api_endpoint='http://localhost:11434/api/generate', **kwargs):
        self.model_name = model_name
        self.api_endpoint = api_endpoint
        self.session = requests.Session()
        self.kwargs = {"temperature": 0.7, "n": 1, **kwargs}
        print(f"Initialized OLLAMA with model_name: {model_name}, api_endpoint: {api_endpoint}, kwargs: {self.kwargs}")

    def predict(self, question, **kwargs):
        complete_response = ""
        payload = {'model': self.model_name, 'prompt': question, **self.kwargs, **kwargs}
        with self.session.post(self.api_endpoint, json=payload, stream=True) as r:
            if r.status_code == 200:
                for line in r.iter_lines():
                    if line:
                        decoded_line = line.decode('utf-8')
                        json_response = json.loads(decoded_line)
                        complete_response += json_response.get("response", "")
                        if json_response.get("done", False):
                            break
            else:
                print(f"Error: Received status code {r.status_code}")
        return complete_response.strip()

    def set_run_config(self, run_config):
        print("Setting run configuration...")
        self.run_config = run_config
        # Add any necessary setup based on run_config here

# Create an instance of the enhanced OLLAMA class
ollama_client = OLLAMA(model_name="mistral")


Initialized OLLAMA with model_name: mistral, api_endpoint: http://localhost:11434/api/generate, kwargs: {'temperature': 0.7, 'n': 1}


In [36]:
# Initialize the Ollama client
ollama_client = Ollama(model="mistral")

# Example of generating a response
def get_response(question):
    response = ollama_client.chat(question)
    return response.text

class EvaluationCompatibleOllama:
    def __init__(self, ollama_client):
        self.ollama = ollama_client
        self.run_config = None

    def predict(self, question):
        return self.ollama.chat(question).text

    def set_run_config(self, run_config):
        self.run_config = run_config
        # Configure your model based on run_config if necessary


In [50]:
import langchain as lc
#from langchain.llms import Mistral
from qdrant_client import models, QdrantClient
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.vectorstores.qdrant import Qdrant
from langchain.text_splitter import RecursiveCharacterTextSplitter
import PyPDF2

#from langchain.chains import RetrievalQA
# Extract text from PDF
def extract_text_from_pdf(file_path):
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ''
        for page in reader.pages:
            text += page.extract_text()
    return text
texts = extract_text_from_pdf(r'C:\Users\Tifa\Desktop\rag\src\data\circulaire.pdf')

# Create documents with metadata (dummy metadata in this example)
documents = [{'text': text, 'metadata': {'source': 'Medical QA PDF'}} for text in texts]


In [54]:
embeddings = HuggingFaceEmbeddings(model_kwargs = {'device': 'cpu'},
                                    encode_kwargs = {'normalize_embeddings': False})

c:\Users\Tifa\Desktop\rag\.venv\lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tifa\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


KeyboardInterrupt: 

In [55]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embed_model = HuggingFaceEmbeddings(model_name=config["embedding"])


NameError: name 'config' is not defined

In [ ]:
doc_store = Qdrant.from_texts(texts=[doc['text'] for doc in documents],
                                                  metadatas=[doc['metadata'] for doc in documents],
                                                  embedding=embeddings,
                                                  location=":memory:",
                                                  prefer_grpc=True,
                                                  collection="medical_qa_search")

# Initialize the Mistral model
mistral_model = Mistral()

 # Adjust according to how results are structured in LangChain


In [ ]:
# Setup the RetrievalQA pipeline
retrieval_qa = RetrievalQA.from_chain_type(
    llm=mistral_model,
    retriever=doc_store.as_retriever(search_kwargs={"k": 5}),
    prompt_template="{context}\n\nQUESTION:```{question}```\nANSWER:"
)

# Ask a question
question = input("Quelles sont les nouvelles directives pour la mise en œuvre en Tunisie?")
result = retrieval_qa(question)

# Output the result
print(result.answer) 